In [ ]:
!pip install -q --upgrade --force-reinstall "python-telegram-bot>=21.7,<23"

In [ ]:
import telegram
import sys

print("PYTHON:", sys.version)
print("TELEGRAM:", telegram.__version__)
print("RUTA:", telegram.__file__)

PYTHON: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
TELEGRAM: 22.8
RUTA: /usr/local/lib/python3.13/dist-packages/telegram/__init__.py


In [ ]:
!pip install -q gspread google-auth

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving chatbot-para-registro-de-datos-b292d3d251a3.json to chatbot-para-registro-de-datos-b292d3d251a3.json


In [ ]:
import gspread
from google.oauth2.service_account import Credentials

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "chatbot-para-registro-de-datos-b292d3d251a3.json",
    scopes=SCOPES
)

cliente = gspread.authorize(creds)

archivo = cliente.open("Registro de accidentes")

hoja_accidentes = archivo.worksheet("Accidentes")
hoja_volumen = archivo.worksheet("Vol Recolectado")

print("✅ Conexión exitosa")
print("Accidentes:", hoja_accidentes.title)
print("Volumen:", hoja_volumen.title)

✅ Conexión exitosa
Accidentes: Accidentes
Volumen: Vol Recolectado


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from telegram import (
    Update,
    ReplyKeyboardMarkup,
    ReplyKeyboardRemove
)

from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    ConversationHandler,
    filters
)

# =====================================================
# TOKEN DEL BOT
# =====================================================

TOKEN = "8660238888:AAFQ7r1rXuCI-zHBKJI4SNnWn_rdRnZEYsM"

# =====================================================
# ESTADOS
# =====================================================

CAMION = 1
TURNO = 2
BARRIO = 3
TIPO = 4
LESION = 5
CONFIRMAR = 6
KG_ACCIDENTE = 7 # This state is now only used by `conv_volumen` (implicitly, if `vol_kg` is used for accident kg, it needs to be VOL_KG) - keeping for consistency
OBSERVACION = 8 # Define OBSERVACION state
VOL_CAMION = 20
VOL_TURNO = 21
VOL_KG = 22

# =====================================================
# BARRIOS
# =====================================================

BARRIOS = {

    "Compactador":[
        "Palermo",
        "Ampliación Palermo",
        "Municipal",
        "Centro",
        "Panorámico Parte Este",
        "Bella Vista",
        "Belgrano"
    ],

    "Volcador 1":[
        "Malvinas",
        "Foprovi",
        "Fonavi",
        "CGT",
        "Cerro De Las Rosas Norte",
        "Norte",
        "Portal Del Norte"
    ],

    "Volcador 2":[
        "La Merced",
        "Bella Vista",
        "Primero De Mayo",
        "Talleres",
        "La Jalea",
        "Cristo Redentor",
        "Raúl Alfonsín",
        "Francisco Primero",
        "Cerro Del Sur",
        "Tala Viejo",
        "El Tala",
        "Jardín",
        "Blancos Del Sur"
    ]
}

# =====================================================
# TIPOS DE ACCIDENTES
# =====================================================

ACCIDENTES = {

    "Lesiones Físicas Directas": [
        "Cortes",
        "Pinchazos",
        "Golpes",
        "Esguinces"
    ],

    "Caídas y Atropellos": [
        "Resbalones",
        "Caídas desde el camión",
        "Atropellos",
        "Atrapamientos"
    ],

    "Sobrecarga Corporal": [
        "Lumbalgia",
        "Desgarros",
        "Hernias"
    ],

    "Riesgos Ambientales y Químicos": [
        "Quemaduras químicas",
        "Explosiones",
        "Infecciones",
        "Inhalación de gases"
    ]

}

# Define `volumenes` as a global list, as it's used with `.append` later.
volumenes = []
from datetime import datetime

# =====================================================
# GUARDAR EN GOOGLE SHEETS
# =====================================================

def guardar_accidente(context):

    fecha = datetime.now().strftime("%d/%m/%Y %H:%M")

    hoja_accidentes.append_row([
        fecha,
        context.user_data["camion"],
        context.user_data["ruta"],
        context.user_data["turno"],
        context.user_data["barrio"],
        context.user_data["tipo"],
        context.user_data["lesion"],
        context.user_data["observacion"]
    ])
from datetime import datetime

def guardar_volumen(context):

    fecha = datetime.now().strftime("%d/%m/%Y %H:%M")

    hoja_volumen.append_row([
        fecha,
        context.user_data["vol_camion"],
        context.user_data["vol_turno"],
        context.user_data["vol_kg"]
    ])

# =====================================================
# START
# =====================================================

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):

    teclado = [
        ["📝 Nuevo Accidente"],
        ["⚖️ Vol recolectados"],
        ["📊 Estadísticas"],
        ["📁 Ver Registros"]
    ]

    await update.message.reply_text(

        "👋 Bienvenido al Sistema de Registro de Datos\n\n"
        "Seleccione una opción.",

        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True
        )
    )

# =====================================================
# NUEVO ACCIDENTE
# =====================================================

async def nuevo_accidente(update: Update, context: ContextTypes.DEFAULT_TYPE):

    teclado = [
        ["Compactador"],
        ["Volcador 1"],
        ["Volcador 2"]
    ]

    await update.message.reply_text(

        "🚛 Seleccione el camión:",

        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return CAMION

# =====================================================
# CAMION
# =====================================================

async def seleccionar_camion(update: Update, context: ContextTypes.DEFAULT_TYPE):

    camion = update.message.text

    context.user_data["camion"] = camion

    teclado = [
        ["Mañana"],
        ["Tarde"]
    ]

    await update.message.reply_text(

        "🕒 Seleccione el turno:",

        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return TURNO

# =====================================================
# TURNO
# =====================================================

async def seleccionar_turno(update: Update, context: ContextTypes.DEFAULT_TYPE):

    turno = update.message.text

    context.user_data["turno"] = turno

    camion = context.user_data["camion"]

    barrios = BARRIOS[camion]

    teclado = []

    for barrio in barrios:

        teclado.append([barrio])

    await update.message.reply_text(

        "🏘️ Seleccione el barrio:",

        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return BARRIO

# =====================================================
# BARRIO
# =====================================================
async def seleccionar_barrio(update: Update, context: ContextTypes.DEFAULT_TYPE):

    barrio = update.message.text

    context.user_data["barrio"] = barrio

    camion = context.user_data["camion"]

    if camion == "Compactador":
        ruta = "Centro"

    elif camion == "Volcador 1":
        ruta = "Norte"

    else:
        ruta = "Sur"

    context.user_data["ruta"] = ruta

    teclado = []

    for tipo in ACCIDENTES.keys():
        teclado.append([tipo])

    await update.message.reply_text(

        "⚠️ Seleccione el tipo de accidente:",

        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return TIPO

async def seleccionar_tipo(update: Update, context: ContextTypes.DEFAULT_TYPE):

    tipo = update.message.text

    context.user_data["tipo"] = tipo

    lesiones = ACCIDENTES[tipo]

    teclado = []

    for lesion in lesiones:
        teclado.append([lesion])

    await update.message.reply_text(

        "🩹 Seleccione la lesión:",

        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return LESION

async def seleccionar_lesion(update: Update, context: ContextTypes.DEFAULT_TYPE):

    lesion = update.message.text

    context.user_data["lesion"] = lesion

    await update.message.reply_text(

        "📝 Escriba una observación.\n\nSi no desea agregar ninguna escriba:\n\nSin observaciones", # Prompt directly for observation

        reply_markup=ReplyKeyboardRemove()

    )
    return OBSERVACION # Transition directly to OBSERVACION

# Removed ingresar_kg_accidente function as 'kg' is no longer collected in this flow

async def ingresar_observacion(update: Update, context: ContextTypes.DEFAULT_TYPE):

    context.user_data["observacion"] = update.message.text
    guardar_accidente(context)

    resumen = f"""
✅ ACCIDENTE REGISTRADO

🚛 Camión:
{context.user_data['camion']}

🛣️ Ruta:
{context.user_data['ruta']}

🕒 Turno:
{context.user_data['turno']}

🏘️ Barrio:
{context.user_data['barrio']}

⚠️ Tipo:
{context.user_data['tipo']}

🩹 Lesión:
{context.user_data['lesion']}

📝 Observación:
{context.user_data['observacion']}
""" # Removed 'Kg recolectados' from summary

    await update.message.reply_text(
        resumen
    )

    return ConversationHandler.END
# =====================================================
# LEER REGISTROS
# =====================================================

def obtener_ultimos_registros(cantidad=5):

    registros = hoja_accidentes.get_all_values()

    # Elimina la fila de encabezados
    registros = registros[1:]

    # Si no hay registros
    if len(registros) == 0:
        return []

    # Devuelve los últimos registros
    return registros[-cantidad:]
# =====================================================
# VER REGISTROS
# =====================================================

async def ver_registros(update: Update, context: ContextTypes.DEFAULT_TYPE):

    registros = obtener_ultimos_registros()

    if not registros:

        await update.message.reply_text(
            "📁 No existen registros."
        )

        return

    mensaje = "📁 Últimos registros\n\n"

    for i, registro in enumerate(registros, start=1):

        fecha = registro[0]
        camion = registro[1]
        ruta = registro[2]
        turno = registro[3]
        barrio = registro[4]
        tipo = registro[5]
        lesion = registro[6]

        mensaje += (
            f"📌 Registro {i}\n"
            f"📅 Fecha: {fecha}\n"
            f"🚛 Camión: {camion}\n"
            f"🛣 Ruta: {ruta}\n"
            f"🕒 Turno: {turno}\n"
            f"🏘 Barrio: {barrio}\n"
            f"⚠️ Tipo: {tipo}\n"
            f"🩹 Lesión: {lesion}\n"
            f"-------------------------\n\n"
        )

    await update.message.reply_text(mensaje)
# =====================================================
# CANCELAR
# =====================================================

async def cancelar(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text(

        "Operación cancelada.",

        reply_markup=ReplyKeyboardRemove()
    )

    return ConversationHandler.END

# =====================================================
# CONVERSACION
# =====================================================

conv_handler = ConversationHandler(

    entry_points=[
        MessageHandler(
            filters.Regex("^📝 Nuevo Accidente$"),
            nuevo_accidente
        )
    ],

    states={

        CAMION:[
            MessageHandler(filters.TEXT & ~filters.COMMAND, seleccionar_camion)
        ],

        TURNO: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, seleccionar_turno)
        ],

        BARRIO: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, seleccionar_barrio)
        ],

        TIPO: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, seleccionar_tipo)
        ],

        LESION: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, seleccionar_lesion)
        ],

        OBSERVACION: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, ingresar_observacion)
        ]

    }, # Removed KG_ACCIDENTE state

    fallbacks=[
        CommandHandler(
            "cancel",
            cancelar
        )
    ]

)
async def iniciar_volumen(update: Update, context: ContextTypes.DEFAULT_TYPE):

    teclado = [
        ["Compactador"],
        ["Volcador 1"],
        ["Volcador 2"]
    ]

    await update.message.reply_text(
        "🚛 Seleccione el camión:",
        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return VOL_CAMION
async def volumen_camion(update: Update, context: ContextTypes.DEFAULT_TYPE):

    context.user_data["vol_camion"] = update.message.text

    teclado = [
        ["Mañana"],
        ["Tarde"]
    ]

    await update.message.reply_text(
        "🕒 Seleccione el turno:",
        reply_markup=ReplyKeyboardMarkup(
            teclado,
            resize_keyboard=True,
            one_time_keyboard=True
        )
    )

    return VOL_TURNO
async def volumen_turno(update: Update, context: ContextTypes.DEFAULT_TYPE):

    context.user_data["vol_turno"] = update.message.text

    await update.message.reply_text(
        "⚖️ Ingrese los Kg recolectados:",
        reply_markup=ReplyKeyboardRemove()
    )

    return VOL_KG

async def volumen_kg(update: Update, context: ContextTypes.DEFAULT_TYPE):

    try:
        kg = float(update.message.text)
        context.user_data["vol_kg"] = kg
        guardar_volumen(context)
    except ValueError:
        await update.message.reply_text(
            "Ingrese un número válido."
        )
        return VOL_KG

    registro = {

        "Camión": context.user_data["vol_camion"],
        "Turno": context.user_data["vol_turno"],
        "Kg": kg

    }

    volumenes.append(registro)

    await update.message.reply_text(

        f"""✅ Volumen registrado

🚛 Camión: {registro['Camión']}
🕒 Turno: {registro['Turno']}
⚖️ Kg: {registro['Kg']} Kg"""

    )

    return ConversationHandler.END
conv_volumen = ConversationHandler(

    entry_points=[
        MessageHandler(
            filters.Regex("^⚖️ Vol recolectados$"),
            iniciar_volumen
        )
    ],

    states={

        VOL_CAMION: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, volumen_camion)
        ],

        VOL_TURNO: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, volumen_turno)
        ],

        VOL_KG: [
            MessageHandler(filters.TEXT & ~filters.COMMAND, volumen_kg)
        ]

    },

    fallbacks=[
        CommandHandler(
            "cancel",
            cancelar
        )
    ]

)

# Placeholder for error handler, to prevent NameError
async def error(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print(f"Update {update} caused error {context.error}")

# =====================================================
# BOT
# =====================================================

app = ApplicationBuilder().token(TOKEN).build()

app.add_handler(CommandHandler("start", start))
app.add_handler(
    MessageHandler(
        filters.Regex("^📁 Ver Registros$"),
        ver_registros
    )
)


app.add_handler(conv_handler)
app.add_handler(conv_volumen)
app.add_error_handler(error)

print("✅ BOT INICIADO")

app.run_polling()

✅ BOT INICIADO
Update None caused error httpx.ReadError: 


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# ==========================================
# BARRIOS POR CAMIÓN
# ==========================================

BARRIOS = {
    "Compactador": [
        "Palermo",
        "Ampliación Palermo",
        "Municipal",
        "Centro",
        "Panorámico Parte Este",
        "Bella Vista",
        "Belgrano"
    ],

    "Volcador 1": [
        "Malvinas",
        "Foprovi",
        "Fonavi",
        "CGT",
        "Cerro De Las Rosas Norte",
        "Norte",
        "Portal Del Norte"
    ],

    "Volcador 2": [
        "La Merced",
        "Bella Vista",
        "Primero De Mayo",
        "Talleres",
        "La Jalea",
        "Cristo Redentor",
        "Raúl Alfonsín",
        "Francisco Primero",
        "Cerro Del Sur",
        "Tala Viejo",
        "El Tala",
        "Jardín",
        "Blancos Del Sur"
    ]
}

# ==========================================
# TIPOS DE ACCIDENTES
# ==========================================

ACCIDENTES = {
    "Lesiones Físicas Directas": [
        "Cortes",
        "Pinchazos",
        "Golpes",
        "Esguinces"
    ],

    "Caídas y Atropellos": [
        "Resbalones",
        "Caídas desde el camión",
        "Atropellos",
        "Atrapamientos"
    ],

    "Sobrecarga Corporal": [
        "Lumbalgia",
        "Desgarros",
        "Hernias"
    ],

    "Riesgos Ambientales y Químicos": [
        "Quemaduras químicas",
        "Explosiones",
        "Infecciones",
        "Inhalación de gases"
    ]
}

CAMIONES = list(BARRIOS.keys())
TURNOS = ["Mañana", "Tarde"]

# Convertimos el rango de fechas directamente a una lista de strings para evitar conflictos con numpy.datetime64
FECHAS = pd.date_range(
    start="2026-09-01",
    end="2026-09-30"
).strftime("%Y-%m-%d").tolist()

# ==========================================
# 120 REGISTROS DE RESIDUOS
# ==========================================

residuos = []

for i in range(1, 121):

    camion = np.random.choice(CAMIONES)
    barrio = np.random.choice(BARRIOS[camion])

    residuos.append({

        "fecha": np.random.choice(FECHAS),
        "turno": np.random.choice(TURNOS),
        "camion": camion,
        "barrio": barrio,
        "volumen_m3": round(np.random.uniform(1, 12), 2),
        "origen_dato": "SIMULADO"
    })

df_residuos = pd.DataFrame(residuos)

# ==========================================
# 12 REGISTROS DE ACCIDENTES
# ==========================================

accidentes = []

for i in range(1, 13):

    categoria = np.random.choice(list(ACCIDENTES.keys()))
    tipo = np.random.choice(ACCIDENTES[categoria])

    camion = np.random.choice(CAMIONES)
    barrio = np.random.choice(BARRIOS[camion])

    accidentes.append({

        "fecha": np.random.choice(FECHAS),
        "turno": np.random.choice(TURNOS),
        "camion": camion,
        "barrio": barrio,
        "categoria_accidente": categoria,
        "tipo_accidente": tipo,
        "gravedad": np.random.choice(
            ["Leve", "Moderado", "Grave"],
            p=[0.60, 0.30, 0.10]
        ),
        "estado": np.random.choice(
            ["Registrado", "En seguimiento", "Cerrado"]
        ),
        "origen_dato": "SIMULADO"
    })

df_accidentes = pd.DataFrame(accidentes)

print("Residuos generados:", len(df_residuos))
print("Accidentes generados:", len(df_accidentes))

display(df_residuos.head())
display(df_accidentes.head())

Residuos generados: 120
Accidentes generados: 12


,fecha,turno,camion,barrio,volumen_m3,origen_dato
0,2026-09-29,Mañana,Volcador 2,Talleres,9.05,SIMULADO
1,2026-09-07,Tarde,Compactador,Panorámico Parte Este,2.72,SIMULADO
2,2026-09-24,Mañana,Volcador 2,El Tala,7.61,SIMULADO
3,2026-09-21,Tarde,Volcador 2,Cristo Redentor,8.94,SIMULADO
4,2026-09-02,Tarde,Volcador 1,Norte,11.91,SIMULADO


,fecha,turno,camion,barrio,categoria_accidente,tipo_accidente,gravedad,estado,origen_dato
0,2026-09-30,Mañana,Volcador 2,Cerro Del Sur,Sobrecarga Corporal,Lumbalgia,Leve,Registrado,SIMULADO
1,2026-09-17,Tarde,Volcador 2,Primero De Mayo,Caídas y Atropellos,Resbalones,Leve,Registrado,SIMULADO
2,2026-09-14,Mañana,Volcador 2,Jardín,Sobrecarga Corporal,Lumbalgia,Grave,Cerrado,SIMULADO
3,2026-09-03,Tarde,Compactador,Panorámico Parte Este,Caídas y Atropellos,Atropellos,Leve,Cerrado,SIMULADO
4,2026-09-16,Mañana,Compactador,Belgrano,Lesiones Físicas Directas,Esguinces,Leve,Registrado,SIMULADO


segunndo intento


In [ ]:
import pandas as pd
import numpy as np
import gspread

# =====================================================
# 1. CONFIGURACIÓN
# =====================================================

np.random.seed(42)

# Usar las credenciales existentes del chatbot (actualizado con el archivo correcto)
gc = gspread.service_account(
    filename="chatbot-para-registro-de-datos-b292d3d251a3.json"
)

# Abrir la hoja de cálculo existente
spreadsheet = gc.open("Registro de accidentes")

# Acceder a las pestañas existentes
hoja_residuos = spreadsheet.worksheet("Vol Recolectado")
hoja_accidentes = spreadsheet.worksheet("Accidentes")

# =====================================================
# 2. BARRIOS POR CAMIÓN
# =====================================================

BARRIOS = {
    "Compactador": [
        "Palermo",
        "Ampliación Palermo",
        "Municipal",
        "Centro",
        "Panorámico Parte Este",
        "Bella Vista",
        "Belgrano"
    ],

    "Volcador 1": [
        "Malvinas",
        "Foprovi",
        "Fonavi",
        "CGT",
        "Cerro De Las Rosas Norte",
        "Norte",
        "Portal Del Norte"
    ],

    "Volcador 2": [
        "La Merced",
        "Bella Vista",
        "Primero De Mayo",
        "Talleres",
        "La Jalea",
        "Cristo Redentor",
        "Raúl Alfonsín",
        "Francisco Primero",
        "Cerro Del Sur",
        "Tala Viejo",
        "El Tala",
        "Jardín",
        "Blancos Del Sur"
    ]
}

CAMIONES = list(BARRIOS.keys())
TURNOS = ["Mañana", "Tarde"]

FECHAS = pd.date_range(
    start="2026-09-01",
    end="2026-09-30"
)

# =====================================================
# 3. TIPOS DE ACCIDENTES
# =====================================================

ACCIDENTES = {
    "Lesiones Físicas Directas": [
        "Cortes",
        "Pinchazos",
        "Golpes",
        "Esguinces"
    ],

    "Caídas y Atropellos": [
        "Resbalones",
        "Caídas desde el camión",
        "Atropellos",
        "Atrapamientos"
    ],

    "Sobrecarga Corporal": [
        "Lumbalgia",
        "Desgarros",
        "Hernias"
    ],

    "Riesgos Ambientales y Químicos": [
        "Quemaduras químicas",
        "Explosiones",
        "Infecciones",
        "Inhalación de gases"
    ]
}

# =====================================================
# 4. PROTECCIÓN CONTRA DUPLICADOS
# =====================================================

# Consultar las observaciones actuales
datos_actuales_accidentes = hoja_accidentes.get_all_records()

simulados_ya_cargados = any(
    "SIMULADO-ACC-" in str(fila.get("Observacion", ""))
    for fila in datos_actuales_accidentes
)

if simulados_ya_cargados:
    raise RuntimeError(
        "Ya existen accidentes simulados. "
        "Se detuvo la carga para evitar duplicados. "
        "Revisá las pestañas antes de continuar."
    )

# =====================================================
# 5. GENERAR 120 REGISTROS DE RECOLECCIÓN
# =====================================================

registros_residuos = []

for i in range(1, 121):

    camion = np.random.choice(CAMIONES)
    barrio = np.random.choice(BARRIOS[camion])

    fecha = pd.Timestamp(
        np.random.choice(FECHAS)
    ).strftime("%d/%m/%Y")

    turno = np.random.choice(TURNOS)

    # Kilogramos simulados
    kg = int(np.random.randint(500, 5001))

    registros_residuos.append([
        fecha,
        camion,
        turno,
        kg
    ])

# =====================================================
# 6. GENERAR 12 ACCIDENTES LABORALES
# =====================================================

registros_accidentes = []

for i in range(1, 13):

    camion = np.random.choice(CAMIONES)
    barrio = np.random.choice(BARRIOS[camion])

    fecha = pd.Timestamp(
        np.random.choice(FECHAS)
    ).strftime("%d/%m/%Y")

    turno = np.random.choice(TURNOS)

    categoria = np.random.choice(
        list(ACCIDENTES.keys())
    )

    tipo = np.random.choice(
        ACCIDENTES[categoria]
    )

    registros_accidentes.append([
        fecha,
        camion,
        barrio,  # Ruta
        turno,
        barrio,
        categoria,
        tipo,
        f"SIMULADO-ACC-{i:03d} | "
        f"Registro ficticio para pruebas. "
        f"Tipo: {tipo}. Categoría: {categoria}."
    ])

# =====================================================
# 7. CARGAR LOS REGISTROS EN LAS PESTAÑAS EXISTENTES
# =====================================================

# Leer los registros de residuos para revisar si la pestaña
# contiene datos. No se borra ni modifica nada existente.
datos_actuales_residuos = hoja_residuos.get_all_records()

print("Registros actuales de residuos:", len(datos_actuales_residuos))
print("Nuevos registros de residuos:", len(registros_residuos))
print("Nuevos accidentes:", len(registros_accidentes))

# Agregar datos al final de cada pestaña
# Los encabezados existentes no se modifican.

hoja_residuos.append_rows(
    registros_residuos,
    value_input_option="USER_ENTERED"
)

hoja_accidentes.append_rows(
    registros_accidentes,
    value_input_option="USER_ENTERED"
)

print("¡Carga completada!")
print("Se agregaron 120 registros simulados de residuos.")
print("Se agregaron 12 registros simulados de accidentes.")

Registros actuales de residuos: 15
Nuevos registros de residuos: 120
Nuevos accidentes: 12
¡Carga completada!
Se agregaron 120 registros simulados de residuos.
Se agregaron 12 registros simulados de accidentes.


In [ ]:
import gspread

gc = gspread.service_account(
    filename="chatbot-para-registro-de-datos-b292d3d251a3.json"
)

spreadsheet = gc.open("Registro de accidentes")

hoja_residuos = spreadsheet.worksheet("Vol Recolectado")
hoja_accidentes = spreadsheet.worksheet("Accidentes")

print("Columnas de Vol Recolectado:")
print(hoja_residuos.row_values(1))

print("\nColumnas de Accidentes:")
print(hoja_accidentes.row_values(1))

Columnas de Vol Recolectado:
['Fecha', 'Camion ', 'Turno ', 'Kg Recolectados']

Columnas de Accidentes:
['Fecha ', 'Camion ', 'Ruta', 'Turno', 'Barrio', 'Tipo', 'Lesion ', 'Observacion']


In [ ]:
!pip install -q streamlit
import pandas as pd
import gspread
import streamlit as st

# Usamos service_account directo con el archivo local en lugar de st.secrets
gc = gspread.service_account(
    filename="chatbot-para-registro-de-datos-b292d3d251a3.json"
)

spreadsheet = gc.open("Registro de accidentes")

hoja_residuos = spreadsheet.worksheet("Vol Recolectado")
hoja_accidentes = spreadsheet.worksheet("Accidentes")


@st.cache_data(ttl=30)
def cargar_datos():

    registros_residuos = hoja_residuos.get_all_records()
    registros_accidentes = hoja_accidentes.get_all_records()

    df_residuos = pd.DataFrame(registros_residuos)
    df_accidentes = pd.DataFrame(registros_accidentes)

    return df_residuos, df_accidentes


df_residuos, df_accidentes = cargar_datos()

st.title("♻️ Dashboard de Obras y Servicios")

st.info(
    "ENTORNO DE PRUEBA: contiene datos simulados "
    "y puede incluir registros reales ingresados por el chatbot."
)

col1, col2, col3 = st.columns(3)

with col1:
    st.metric(
        "Registros de residuos",
        len(df_residuos)
    )

with col2:
    st.metric(
        "Volumen registrado",
        f"{pd.to_numeric(df_residuos.get('Volumen', pd.Series(dtype=float)), errors='coerce').sum():.2f} m³"
        if "Volumen" in df_residuos.columns
        else "Revisar columna"
    )

with col3:
    st.metric(
        "Accidentes registrados",
        len(df_accidentes)
    )

st.subheader("Registros de recolección")
st.dataframe(df_residuos, use_container_width=True)

st.subheader("Registros de accidentes")
st.dataframe(df_accidentes, use_container_width=True)

2026-10-16 11:47:11.708 No runtime found, using MemoryCacheStorageManager
2026-10-16 11:47:11.710 No runtime found, using MemoryCacheStorageManager
2026-10-16 11:47:11.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 11:47:12.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 11:47:12.172 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-16 11:47:12.173 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 11:47:12.173 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 11:47:12.175 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-16 11:47:12.176 Thread 'MainThread': missing ScriptRunContext! This warning can be

DeltaGenerator()